In [ ]:
!pip install transformers evaluate tqdm


In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch
import evaluate
from tqdm import tqdm

In [ ]:
model_name = "facebook/bart-large-xsum"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)
device = "cuda" if torch.cuda.is_available() else "cpu"
if(device == "cude"):
  print("running on cuda")
model.to(device)
model.eval()


In [ ]:
!pip install datasets
# You might also need to downgrade huggingface_hub
from datasets import load_dataset

In [ ]:
from datasets import load_dataset
# Use the revision that points to the new Parquet-based files
dataset = load_dataset("EdinburghNLP/xsum", revision="main")
# The 'main' revision usually points to the latest, fixed version now.
# Note: For some datasets, you might need revision="refs/convert/parquet"
# but "main" should work for xsum now.
print(f"Loaded {len(dataset)} samples for evaluation.")

In [ ]:
!pip install rouge_score
rouge = evaluate.load("rouge")

In [ ]:
generated_summaries = []
references = []

for example in tqdm(dataset["test"], desc="Evaluating"):
    article = example["document"]
    reference = example["summary"]

    # Tokenize input
    inputs = tokenizer(article, max_length=1024, truncation=True, return_tensors="pt").to(device)

    # Generate summary
    with torch.no_grad():
        summary_ids = model.generate(
            **inputs,
            num_beams=4,
            length_penalty=2.0,
            max_length=64,
            min_length=11,
            no_repeat_ngram_size=3,
        )

    summary = tokenizer.decode(summary_ids[0], skip_special_tokens=True)

    generated_summaries.append(summary)
    references.append(reference)

In [ ]:
results = rouge.compute(predictions=generated_summaries, references=references, use_stemmer=True)
# saved_data["rouge_scores"] = results
import json
output_file = "rouge_scores.json"
with open(output_file, "w") as f:
    json.dump(results, f, indent=2)

print("\n=== Final ROUGE Results (full XSum test set) ===")
for key, value in results.items():
    print(f"{key}: {value:.4f}")

print(f"\n✅ Full results saved to {output_file}")